# 03 — Model Evaluation

Full evaluation report on the held-out test split:
- Accuracy, macro/weighted F1, per-class precision/recall
- Confusion matrix
- Comparison vs a TF-IDF + Logistic Regression baseline (to show the transformer's value)
- Error analysis: which sentences trip up the model?

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

from src.config import LABEL_LIST, REPORTS_DIR, FIGURES_DIR
from src.data import load_phrasebank, build_splits
from src.predict import SentimentPredictor

sns.set_theme(style='whitegrid')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = load_phrasebank()
splits = build_splits(df)
test = splits['test'].to_pandas()
print(f"Test set: {len(test):,} sentences")

## Fine-tuned DistilBERT — test set predictions

In [ ]:
predictor = SentimentPredictor()
print(f'Loaded: {predictor.source}')

predictions = predictor.predict_batch(test['text'].tolist(), batch_size=32)
label_to_id = {l: i for i, l in enumerate(LABEL_LIST)}
y_pred = np.array([label_to_id[p.label] for p in predictions])
y_true = test['label'].values

In [ ]:
print(f'Accuracy:    {accuracy_score(y_true, y_pred):.4f}')
print(f'F1 (macro):    {f1_score(y_true, y_pred, average="macro"):.4f}')
print(f'F1 (weighted): {f1_score(y_true, y_pred, average="weighted"):.4f}')
print()
print(classification_report(y_true, y_pred, target_names=LABEL_LIST, digits=4))

## Confusion matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, normalize='true')

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.2%', cmap='Blues', square=True,
            xticklabels=LABEL_LIST, yticklabels=LABEL_LIST, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Normalized confusion matrix — DistilBERT')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Baseline: TF-IDF + Logistic Regression
Shows the value of the transformer vs a classical NLP baseline.

In [ ]:
train = splits['train'].to_pandas()
baseline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])
baseline.fit(train['text'], train['label'])
y_pred_baseline = baseline.predict(test['text'])

print('=== TF-IDF + Logistic Regression baseline ===')
print(f'Accuracy:    {accuracy_score(y_true, y_pred_baseline):.4f}')
print(f'F1 (macro):    {f1_score(y_true, y_pred_baseline, average="macro"):.4f}')
print()
print(classification_report(y_true, y_pred_baseline, target_names=LABEL_LIST, digits=4))

## DistilBERT vs Baseline summary

In [ ]:
summary = pd.DataFrame({
    'TF-IDF + LR': [
        accuracy_score(y_true, y_pred_baseline),
        f1_score(y_true, y_pred_baseline, average='macro'),
        f1_score(y_true, y_pred_baseline, average='weighted'),
    ],
    'DistilBERT (fine-tuned)': [
        accuracy_score(y_true, y_pred),
        f1_score(y_true, y_pred, average='macro'),
        f1_score(y_true, y_pred, average='weighted'),
    ],
}, index=['Accuracy', 'F1 macro', 'F1 weighted']).round(4)

print(summary.to_string())
summary.to_csv(REPORTS_DIR / 'model_comparison.csv')

fig, ax = plt.subplots(figsize=(8, 4))
summary.plot.bar(ax=ax, color=['#95A5A6', '#3498DB'])
ax.set_title('DistilBERT vs TF-IDF baseline on Financial PhraseBank')
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Error analysis — which sentences trip up the model?

In [ ]:
test_copy = test.copy()
test_copy['pred'] = y_pred
test_copy['confidence'] = [p.confidence for p in predictions]
test_copy['correct'] = test_copy['label'] == test_copy['pred']

errors = test_copy[~test_copy['correct']].sort_values('confidence', ascending=False).head(15)
print(f'Total errors: {(~test_copy["correct"]).sum():,} / {len(test_copy):,}')
print('\nTop 15 high-confidence errors:')
for _, row in errors.iterrows():
    print(f"  [true={LABEL_LIST[row['label']]} | pred={LABEL_LIST[row['pred']]} | conf={row['confidence']:.2f}]")
    print(f"    \"{row['text'][:160]}\"")